このページではTF-IDFとロジスティック回帰を使って英文フェイクニュースの識別を行う

作成した変数を取り出す

In [2]:
import pickle

with open("../../Data/train_test_split.pkl", "rb") as f:
    x_train, x_test, y_train, y_test = pickle.load(f)

追加の前処理としてNormalizeを行う

In [4]:
import re

def NormalizeText(text):
    text = text.lower() #小文字化
    text = text.replace('\n', ' ') #改行をスペース変換
    text = re.sub(r'[^\w\s]', '', text) #特殊文字を削除
    text = text.strip() #テキストの前後の空白を消去
    return text

x_train_Normalized = x_train.apply(NormalizeText)
x_test_Normalized = x_test.apply(NormalizeText)

TF-IDFを使ってデータを変換する

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

def Tfidf_Limit(limit, x_train, x_test):
    vectorizer = TfidfVectorizer(max_features = limit)
    x_train_tfidf = vectorizer.fit_transform(x_train)
    x_test_tfidf = vectorizer.transform(x_test)
    return x_train_tfidf, x_test_tfidf

x_train_tfidf, x_test_tfidf = Tfidf_Limit(5000, x_train_Normalized, x_test_Normalized)

ロジスティック回帰を使って学習

In [6]:
from sklearn.linear_model import LogisticRegression

def Learning(x_train_tfidf, y_train):
    model = LogisticRegression(max_iter=2000)
    model.fit(x_train_tfidf, y_train)
    return model

model = Learning(x_train_tfidf, y_train)

特徴量5000の時の結果を表示

In [7]:
from sklearn.metrics import classification_report, accuracy_score

def DispResult(model, x_test_tfidf, y_test):
    pred = model.predict(x_test_tfidf)
    print(classification_report(y_test, pred))
    print(accuracy_score(y_test, pred))
    
DispResult(model, x_test_tfidf, y_test)

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4704
           1       0.98      0.99      0.99      4276

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

0.9881959910913141
